# DAAC Actor-Critic — Advantage-Based Top-K Patch Selection
# Dataset: **CIFAR100** (100 classes, 32×32)

**Architecture:** Decoupled Actor-Critic with single-step advantage updates.

| Component | Role |
|---|---|
| Actor | Policy network → per-patch logits → Top-K mask |
| Critic | Value network → scalar V(s) for advantage baseline |
| Advantage | A = R − V(s).detach() |
| Actor loss | −Σ log_prob(selected) · A |
| Critic loss | MSE(V(s), R) |
| Reward | α·min(L₀/Lₜ, 3.0) − (1−α)·(K/N) |

Patch grid: 8×8 = 64 patches  |  K selected = 40 (62%)

**Table 1** — Val Loss | Accuracy | Precision | Recall | F1 | Epoch Time | Patches | Patch %
**Table 2** — Train GFLOPs | Infer GFLOPs | FPS | CPU Mem | GPU Mem | Peak GPU Mem
**Table 3** — Mean & Best summary


In [ ]:
!pip install -q einops vit_pytorch fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

models_save_path        = '/content/drive/MyDrive/Tirocinio/Finale/Models/'
results_save_path_agent = '/content/drive/MyDrive/Tirocinio/Finale/Results/'
dataset_path            = '/content/drive/MyDrive/Tirocinio/Transformers/Datasets/'

import os
os.makedirs(models_save_path,        exist_ok=True)
os.makedirs(results_save_path_agent, exist_ok=True)

Mounted at /content/drive


In [ ]:
import numpy as np
import math, time, os, random
from collections import deque
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms

from einops import rearrange
from einops.layers.torch import Rearrange

import matplotlib.pyplot as plt
import pandas as pd
import psutil

from sklearn.metrics import precision_score, recall_score, f1_score

try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except ImportError:
    FVCORE_AVAILABLE = False

def set_seed(seed=42):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

## 1. Profiling Utilities

In [ ]:
def compute_vit_gflops_analytical(dim, depth, heads, mlp_dim, n_active_patches, dim_head=64):
    P = n_active_patches
    inner = dim_head * heads
    pe        = 2 * P * (3*4*4) * dim
    attn_qkv  = 2 * P * dim * (inner * 3)
    attn_dots = 2 * heads * P * P * dim_head
    attn_vals = 2 * heads * P * P * dim_head
    attn_out  = 2 * P * inner * dim
    ff        = 2*P*dim*mlp_dim + 2*P*mlp_dim*dim
    layer     = attn_qkv + attn_dots + attn_vals + attn_out + ff
    return (pe + depth*layer + 2*dim*10) / 1e9

def get_cpu_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def get_gpu_mem_mb(device):
    if device.type != 'cuda': return 0.0, 0.0
    cur  = torch.cuda.memory_allocated(device)    / 1024**2
    peak = torch.cuda.max_memory_reserved(device) / 1024**2
    torch.cuda.reset_peak_memory_stats(device)
    return cur, peak

def measure_fps(model, loader, device, n_batches=10):
    model.eval()
    it = iter(loader); total = 0
    for _ in range(min(3, len(loader))):
        imgs, _ = next(it)
        with torch.no_grad(): _ = model(imgs.to(device))
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(min(n_batches, len(loader))):
        try: imgs, _ = next(it)
        except StopIteration: break
        with torch.no_grad(): _ = model(imgs.to(device))
        total += imgs.size(0)
    if device.type == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return total / elapsed if elapsed > 0 else 0.0

## 2. Vision Transformer

In [ ]:
def pair(t): return t if isinstance(t, tuple) else (t, t)

def posemb_sincos_2d(patches, temperature=10000):
    _, h, w, dim, device, dtype = *patches.shape, patches.device, patches.dtype
    assert dim % 4 == 0
    y, x = torch.meshgrid(torch.arange(h, device=device),
                          torch.arange(w, device=device), indexing='ij')
    omega = 1. / (temperature ** (torch.arange(dim//4, device=device) / (dim//4 - 1)))
    y = y.flatten()[:, None] * omega[None, :]
    x = x.flatten()[:, None] * omega[None, :]
    return torch.cat((x.sin(), x.cos(), y.sin(), y.cos()), dim=1).type(dtype)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0.1):
        super().__init__()
        inner = dim_head * heads
        self.heads, self.scale = heads, dim_head**-0.5
        self.norm = nn.LayerNorm(dim); self.attend = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)
        self.to_qkv = nn.Linear(dim, inner*3, bias=False)
        self.to_out = nn.Linear(inner, dim, bias=False)
    def forward(self, x):
        x = self.norm(x)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d)->b h n d', h=self.heads),
                      self.to_qkv(x).chunk(3, dim=-1))
        attn = self.dropout(self.attend(torch.matmul(q, k.transpose(-1,-2)) * self.scale))
        return self.to_out(rearrange(torch.matmul(attn, v), 'b h n d->b n (h d)'))

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleList([Attention(dim, heads, dim_head, dropout),
                           FeedForward(dim, mlp_dim, dropout)])
            for _ in range(depth)])
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x)+x; x = ff(x)+x
        return x

class PPOViT(nn.Module):
    def __init__(self, *, image_size, patch_size, num_classes, dim,
                 depth, heads, mlp_dim, channels=3, dim_head=64, dropout=0.1):
        super().__init__()
        ih, iw = pair(image_size); ph, pw = pair(patch_size)
        self.num_patches = (ih//ph) * (iw//pw)
        self.patch_grid  = ih//ph
        self.patch_mask  = [1] * self.num_patches
        patch_dim = channels * ph * pw
        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2)->b h w (p1 p2 c)', p1=ph, p2=pw),
            nn.LayerNorm(patch_dim), nn.Linear(patch_dim, dim), nn.LayerNorm(dim))
        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)
        self.linear_head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, num_classes))

    def set_patch_mask(self, mask):
        assert len(mask)==self.num_patches and sum(mask)>0
        self.patch_mask = mask
    def get_patch_mask(self): return self.patch_mask

    def forward(self, img):
        x  = self.to_patch_embedding(img)
        pe = posemb_sincos_2d(x)
        x  = rearrange(x, 'b ... d->b (...) d') + pe
        msk = torch.tensor(self.patch_mask, dtype=torch.bool, device=x.device)
        x  = x[:, msk, :]
        x  = self.transformer(x)
        return self.linear_head(x.mean(dim=1))

    @torch.no_grad()
    def get_context(self, img):
        """Returns (B, P, dim) — per-patch embeddings for the PPO actor-critic."""
        x  = self.to_patch_embedding(img)
        pe = posemb_sincos_2d(x)
        x  = rearrange(x, 'b ... d->b (...) d') + pe
        return self.transformer(x)

## 3. Actor-Critic

In [ ]:
class DAACActorCritic(nn.Module):
    """
    DAAC: Decoupled Actor-Advantage Critic.

    Actor  : dedicated trunk+head → per-patch logits (B, N)
    Critic : dedicated trunk+head → scalar V(s)       (B, 1)

    No shared trunk → zero gradient interference.
    """
    def __init__(self, dim, hidden_dim=256, n_patches=64):
        super().__init__()
        self.n_patches = n_patches

        # ── Actor: context (B,P,dim) → per-patch logits (B,P) ────────
        self.actor_trunk = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.actor_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Linear(hidden_dim//2, 1))   # per-patch logit → squeeze → (B,P)

        # ── Critic: context (B,P,dim) → scalar V(s) (B,1) ────────────
        self.critic_trunk = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.critic_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Linear(hidden_dim//2, 1))   # scalar V(s)

    # ── Actor forward: returns policy logits (B, N) ──────────────────
    def actor_forward(self, context):
        """context (B,P,dim) → policy logits (B,P)"""
        h = self.actor_trunk(context)               # (B,P,hidden)
        return self.actor_head(h).squeeze(-1)        # (B,P)

    # ── Critic forward: returns state value V(s) (B,1) ───────────────
    def critic_forward(self, context):
        """context (B,P,dim) → value V(s) (B,1)"""
        h = self.critic_trunk(context)               # (B,P,hidden)
        return self.critic_head(h.mean(dim=1))       # (B,1) — pool over patches

    def forward(self, context):
        """Returns (logits (B,P), value (B,1)) — unified interface."""
        return self.actor_forward(context), self.critic_forward(context)

    # ── Top-K deterministic mask from policy ─────────────────────────
    def topk_mask(self, context, K):
        """
        policy = actor(state)              # (B, N)
        _, indices = topk(policy, K)
        mask.scatter_(1, indices, 1.0)
        Returns mask (B, N) with exactly K ones per row.
        """
        policy = self.actor_forward(context)          # (B, N)
        _, indices = torch.topk(policy, K, dim=1)     # (B, K)
        mask = torch.zeros_like(policy)
        mask.scatter_(1, indices, 1.0)                # (B, N) — exactly K ones
        return mask, policy

    def actor_parameters(self):
        return list(self.actor_trunk.parameters()) + list(self.actor_head.parameters())

    def critic_parameters(self):
        return list(self.critic_trunk.parameters()) + list(self.critic_head.parameters())


## 4. Rollout Buffer with GAE

In [ ]:
class RolloutBuffer:
    """
    Single-step on-policy buffer.

    Stores (context, mask, log_prob, reward, value) tuples.
    compute_gae() computes GAE advantages for PPO mode.
    compute_single_step_advantages() computes A = R - V.detach() directly
    for the DAAC single-step advantage update (no Bellman, no temporal chain).
    """
    def __init__(self): self.clear()

    def clear(self):
        self.contexts  = []; self.masks     = []
        self.log_probs = []; self.rewards   = []
        self.values    = []; self.dones     = []

    def push(self, context, mask, log_prob, reward, value, done=False):
        self.contexts.append(context.detach().cpu())
        self.masks.append(mask.detach().cpu())
        self.log_probs.append(log_prob.detach().cpu())
        self.rewards.append(float(reward))
        self.values.append(value.detach().cpu())
        self.dones.append(done)

    def compute_gae(self, last_value, gamma=0.99, lam=0.95):
        """GAE advantages — used when rollout_len > 1."""
        T   = len(self.rewards)
        adv = torch.zeros(T)
        ret = torch.zeros(T)
        gae = 0.0
        vals = [v.mean().item() for v in self.values]
        for t in reversed(range(T)):
            nv   = last_value if t == T-1 else vals[t+1]
            dm   = 0.0 if self.dones[t] else 1.0
            d    = self.rewards[t] + gamma*nv*dm - vals[t]
            gae  = d + gamma*lam*dm*gae
            adv[t] = gae
            ret[t] = gae + vals[t]
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        return adv, ret

    def compute_single_step_advantages(self):
        """
        Single-step DAAC advantage: A = R - V.detach()
        No temporal transitions. No Bellman update.
        Returns advantages (T,) and returns (T,) == rewards.
        """
        T    = len(self.rewards)
        R    = torch.tensor(self.rewards, dtype=torch.float)          # (T,)
        V    = torch.stack([v.mean() for v in self.values])           # (T,)
        A    = R - V.detach()                                         # (T,)
        A    = (A - A.mean()) / (A.std() + 1e-8)                     # normalise
        return A, R   # returns == raw rewards (single-step target for critic)

    def __len__(self): return len(self.rewards)


## 5. PPO Agent

In [ ]:
class DAACAgent:
    """
    DAAC Agent — Advantage-based Actor-Critic patch selector.

    Policy (actor):
        policy = actor(state)                        # (B, N)
        _, indices = topk(policy, K, dim=1)
        mask.scatter_(1, indices, 1.0)               # Top-K deterministic

    Reward (stabilised):
        R_acc = min(L0 / Lt, 3.0)
        R     = alpha * R_acc - (1-alpha) * (K/N)

    Advantage (single-step, no Bellman):
        A = R - V(s).detach()

    Actor loss:
        log_probs          = log_softmax(policy, dim=1)
        selected_log_probs = (log_probs * mask).sum(dim=1)
        actor_loss         = -(selected_log_probs * A).mean()

    Critic loss:
        critic_loss = MSE(V(s), R)

    Total loss:
        loss = actor_loss + critic_loss

    Two separate optimisers — zero gradient interference (DAAC).
    """
    def __init__(self, n_patches, att_dim, n_select, hidden_dim=256,
                 actor_lr=3e-4, critic_lr=1e-3,
                 clip_eps=0.2, vf_coef=0.5, ent_coef=0.01,
                 ppo_epochs=4, minibatch=16, lam=0.95, gamma=0.99,
                 beta_ucb=0.5, max_kl=0.015, alpha=0.7, device='cpu'):
        self.n_select   = n_select
        self.n_patches  = n_patches
        self.clip_eps   = clip_eps
        self.vf_coef    = vf_coef
        self.ent_coef   = ent_coef
        self.ppo_epochs = ppo_epochs
        self.minibatch  = minibatch
        self.lam        = lam
        self.gamma      = gamma
        self.beta_ucb   = beta_ucb
        self.max_kl     = max_kl
        self.alpha      = alpha   # reward blending: α·R_acc − (1−α)·(K/N)
        self.device     = device
        self.t          = 0
        self.L0         = None    # baseline loss for reward normalisation

        self.ac = DAACActorCritic(att_dim, hidden_dim, n_patches).to(device)

        # ── Two separate optimisers — the DAAC decoupling ─────────────
        self.actor_optimizer  = optim.AdamW(
            self.ac.actor_parameters(),  lr=actor_lr,  weight_decay=1e-4)
        self.critic_optimizer = optim.AdamW(
            self.ac.critic_parameters(), lr=critic_lr, weight_decay=1e-4)

        self.actor_scheduler  = optim.lr_scheduler.CosineAnnealingLR(
            self.actor_optimizer,  T_max=20000, eta_min=1e-5)
        self.critic_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.critic_optimizer, T_max=20000, eta_min=1e-5)

        self.arm_counts  = torch.zeros(n_patches, device=device)
        self.arm_rewards = torch.zeros(n_patches, device=device)
        self.buffer      = RolloutBuffer()

        self.loss_history        = []
        self.actor_loss_history  = []
        self.critic_loss_history = []

    # ── Stabilised reward ─────────────────────────────────────────────
    def compute_reward(self, current_loss):
        """
        R_acc = min(L0 / Lt, 3.0)      ← clipped ratio, prevents reward explosion
        R     = alpha * R_acc - (1-alpha) * (K/N)   ← sparsity penalty
        """
        if self.L0 is None:
            self.L0 = max(current_loss, 1e-8)   # set baseline on first call
        R_acc = min(self.L0 / max(current_loss, 1e-8), 3.0)
        R     = self.alpha * R_acc - (1.0 - self.alpha) * (self.n_select / self.n_patches)
        return R

    # ── Top-K deterministic patch selection ──────────────────────────
    @torch.no_grad()
    def select_patches(self, context):
        """
        policy = actor(state)              # (B, N)
        _, indices = topk(policy, K)
        mask.scatter_(1, indices, 1.0)    # exactly K active patches
        Returns (mask_list, mask_t, log_prob, value)
        """
        self.t += 1
        context = context.to(self.device)
        B, P, _ = context.shape

        # Actor: policy logits (B, N)
        policy = self.ac.actor_forward(context)          # (B, N)

        # UCB exploration bonus (dataset-level counts, not Bellman)
        ucb = self.beta_ucb * torch.sqrt(
            torch.log(torch.tensor(self.t + 1., device=self.device))
            / (self.arm_counts + 1))
        scores = policy.mean(dim=0) + ucb               # (N,)

        # Top-K deterministic selection
        _, top_k = torch.topk(scores, self.n_select)
        mask_list = [0] * P
        for idx in top_k.cpu().tolist():
            mask_list[idx] = 1
            self.arm_counts[idx] += 1

        # Build float mask (B, N) for loss computation
        mask_t = torch.zeros(B, P, device=self.device)
        mask_t[:, top_k] = 1.0                           # exactly K ones per row

        # log_prob for actor loss: log_softmax(policy) * mask → selected entries
        log_probs_all      = F.log_softmax(policy, dim=1)    # (B, N)
        selected_log_probs = (log_probs_all * mask_t).sum(dim=1)  # (B,)

        # Critic: V(s)
        value = self.ac.critic_forward(context)          # (B, 1)

        return mask_list, mask_t, selected_log_probs, value

    def store(self, context, mask_t, log_prob, reward, value):
        self.buffer.push(context, mask_t, log_prob, reward, value)
        for i in range(mask_t.shape[-1]):
            if mask_t[0, i].item() > 0.5:
                n = self.arm_counts[i].item()
                self.arm_rewards[i] += (reward - self.arm_rewards[i]) / max(n, 1)

    # ── Advantage-based Actor-Critic update ──────────────────────────
    def update(self, last_value=0.0):
        """
        Single-step DAAC update — no Bellman, no temporal transitions.

        A = R - V(s).detach()

        actor_loss  = -(selected_log_probs * A).mean()
        critic_loss = MSE(V(s), R)
        loss        = actor_loss + critic_loss

        Actor and critic update separately (decoupled optimisers).
        """
        if len(self.buffer) == 0:
            return 0.0

        # Single-step advantages: A = R - V.detach()
        adv, ret = self.buffer.compute_single_step_advantages()

        ctx_all = torch.cat(self.buffer.contexts,  dim=0).to(self.device)
        msk_all = torch.cat(self.buffer.masks,     dim=0).to(self.device)
        T = len(self.buffer)
        B = ctx_all.shape[0] // T
        ret = ret.to(self.device)
        adv = adv.to(self.device)

        total_actor_loss  = 0.0
        total_critic_loss = 0.0
        n_up = 0

        for _ in range(self.ppo_epochs):
            idx = torch.randperm(T)
            for start in range(0, T, self.minibatch):
                mb = idx[start:start + self.minibatch]
                if len(mb) == 0:
                    continue

                mb_ctx = torch.cat([ctx_all[i*B:(i+1)*B] for i in mb], dim=0)
                mb_msk = torch.cat([msk_all[i*B:(i+1)*B] for i in mb], dim=0)
                mb_adv = adv[mb].unsqueeze(1).expand(-1, B).reshape(-1)
                mb_ret = ret[mb].unsqueeze(1).expand(-1, B).reshape(-1)

                # ── Actor update ─────────────────────────────────────
                # policy shape: (mb*B, N)
                policy = self.ac.actor_forward(mb_ctx)

                # log_probs = log_softmax(policy, dim=1)
                log_probs_all = F.log_softmax(policy, dim=1)          # (mb*B, N)

                # selected_log_probs = (log_probs * mask).sum(dim=1)
                selected_log_probs = (log_probs_all * mb_msk).sum(dim=1)  # (mb*B,)

                # actor_loss = -(selected_log_probs * A).mean()
                actor_loss = -(selected_log_probs * mb_adv).mean()

                self.actor_optimizer.zero_grad()
                actor_loss.backward()
                nn.utils.clip_grad_norm_(self.ac.actor_parameters(), 0.5)
                self.actor_optimizer.step()
                self.actor_scheduler.step()

                # ── Critic update ────────────────────────────────────
                # critic_loss = MSE(V(s), R)
                V = self.ac.critic_forward(mb_ctx).squeeze(-1)        # (mb*B,)
                critic_loss = F.mse_loss(V, mb_ret)

                self.critic_optimizer.zero_grad()
                critic_loss.backward()
                nn.utils.clip_grad_norm_(self.ac.critic_parameters(), 0.5)
                self.critic_optimizer.step()
                self.critic_scheduler.step()

                total_actor_loss  += actor_loss.item()
                total_critic_loss += critic_loss.item()
                n_up += 1

        self.buffer.clear()
        avg_actor  = total_actor_loss  / max(n_up, 1)
        avg_critic = total_critic_loss / max(n_up, 1)
        total_loss = avg_actor + avg_critic         # loss = actor_loss + critic_loss

        self.actor_loss_history.append(avg_actor)
        self.critic_loss_history.append(avg_critic)
        self.loss_history.append(total_loss)
        return total_loss

    def arm_stats(self):
        return {'counts':  self.arm_counts.cpu().numpy(),
                'rewards': self.arm_rewards.cpu().numpy()}

    def save(self, path):
        torch.save({
            'ac':               self.ac.state_dict(),
            'actor_optimizer':  self.actor_optimizer.state_dict(),
            'critic_optimizer': self.critic_optimizer.state_dict(),
            'arm_counts':       self.arm_counts,
            'arm_rewards':      self.arm_rewards,
            't':                self.t,
            'L0':               self.L0,
        }, path)
        print(f'  DAAC agent saved → {path}')

    def load(self, path):
        ck = torch.load(path, map_location=self.device)
        self.ac.load_state_dict(ck['ac'])
        self.actor_optimizer.load_state_dict(ck['actor_optimizer'])
        self.critic_optimizer.load_state_dict(ck['critic_optimizer'])
        self.arm_counts  = ck['arm_counts'].to(self.device)
        self.arm_rewards = ck['arm_rewards'].to(self.device)
        self.t  = ck['t']
        self.L0 = ck.get('L0', None)


## 6. Environment

In [ ]:
class PPOViTEnv:
    def __init__(self, vit, optimizer, n_patch_selected,
                 loss_weight=5.0, sparsity_weight=1.0, device='cpu'):
        self.vit              = vit
        self.optimizer        = optimizer
        self.n_patch_selected = n_patch_selected
        self.loss_weight      = loss_weight
        self.sparsity_weight  = sparsity_weight
        self.device           = device
        self.train_loss_history = []
        self.train_time_history = []

    def _train_step(self, imgs, labels):
        t0 = time.time(); self.vit.train()
        self.optimizer.zero_grad()
        out  = F.log_softmax(self.vit(imgs), dim=1)
        loss = F.nll_loss(out, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(self.vit.parameters(), 1.0)
        self.optimizer.step()
        return loss.item(), time.time() - t0

    def get_context(self, imgs):
        return self.vit.get_context(imgs)

    def step_train(self, mask, imgs, labels):
        self.vit.set_patch_mask(mask)
        loss, t = self._train_step(imgs, labels)
        self.train_loss_history.append(loss)
        self.train_time_history.append(t)

    def step_reward(self, mask, imgs, labels, agent):
        """
        Stabilised reward:
            R_acc = min(L0 / Lt, 3.0)
            R     = alpha * R_acc - (1-alpha) * (K/N)

        agent.compute_reward() handles L0 initialisation and clipping.
        Returns (new_context, reward, n_selected).
        """
        self.vit.set_patch_mask(mask)
        n_sel = sum(mask)
        loss, t = self._train_step(imgs, labels)
        self.train_loss_history.append(loss)
        self.train_time_history.append(t)

        # Use agent's stabilised reward (not raw loss ratio)
        reward = agent.compute_reward(loss)

        new_ctx = self.get_context(imgs)
        return new_ctx, reward, n_sel


## 7. Dataset

In [ ]:
batch_size   = 128
img_size     = 32
dataset_name = 'CIFAR100_DAAC_'

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))])
transform_val = transforms.Compose([
    transforms.Resize(img_size), transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))])

trainset = torchvision.datasets.CIFAR100(
    root=dataset_path, train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

full_val = torchvision.datasets.CIFAR100(
    root=dataset_path, train=False, download=True, transform=transform_val)
val_n, test_n = int(0.95*len(full_val)), int(0.05*len(full_val))
valset, testset = data.random_split(full_val, [val_n, test_n])
val_loader  = data.DataLoader(valset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

classes = trainset.classes
print(f'Classes: {len(classes)}  |  dataset: CIFAR-100')


Classes: 100  |  dataset: CIFAR-100


## 8. Trainer with Full Metrics

In [ ]:
def measure_fps(vit_model, loader, device, n_batches=10):
    vit_model.eval()
    t0 = time.time(); total = 0
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            if i >= n_batches: break
            imgs = imgs.to(device)
            _ = vit_model(imgs)
            total += imgs.size(0)
    return total / max(time.time() - t0, 1e-6)


class PPOTrainer:
    """
    DAAC Trainer with full metrics.

    Table 1 — quality : Val Loss | Accuracy | Precision | Recall | F1 | Epoch Time | Patches | Patch %
    Table 2 — compute : Train GFLOPs | Infer GFLOPs | FPS | CPU Mem | GPU Mem | Peak GPU Mem
    Table 3 — summary : Mean & Best across all epochs
    """
    def __init__(self, vit, agent, env, train_loader, val_loader,
                 epochs, reward_every, rollout_len,
                 vit_dim, vit_depth, vit_heads, vit_mlp_dim,
                 n_active_patches, device):
        self.vit           = vit
        self.agent         = agent
        self.env           = env
        self.train_loader  = train_loader
        self.val_loader    = val_loader
        self.epochs        = epochs
        self.reward_every  = reward_every
        self.rollout_len   = rollout_len
        self.device        = device
        self.vit_dim       = vit_dim;  self.vit_depth   = vit_depth
        self.vit_heads     = vit_heads; self.vit_mlp_dim = vit_mlp_dim
        self.n_active_patches = n_active_patches
        # Table 1
        self.t1_val_loss=[]; self.t1_val_acc=[]; self.t1_precision=[]
        self.t1_recall=[]; self.t1_f1=[]; self.t1_epoch_time=[]
        self.t1_patches_selected=[]; self.t1_patch_pct=[]
        # Table 2
        self.t2_train_gflops=[]; self.t2_infer_gflops=[]; self.t2_fps=[]
        self.t2_cpu_mem=[]; self.t2_gpu_mem=[]; self.t2_peak_gpu_mem=[]
        # Diagnostics
        self.step_rewards=[]; self.ppo_losses=[]

    def train(self):
        print('='*65)
        print('  DAAC Actor-Critic + ViT  |  Advantage-Based Patch Selection')
        print(f'  alpha={self.agent.alpha}  n_select={self.agent.n_select}/{self.vit.num_patches}')
        print(f'  actor_lr={self.agent.actor_optimizer.param_groups[0]["lr"]}'
              f'  critic_lr={self.agent.critic_optimizer.param_groups[0]["lr"]}')
        print('='*65)

        train_gflops = compute_vit_gflops_analytical(
            self.vit_dim, self.vit_depth, self.vit_heads, self.vit_mlp_dim,
            self.n_active_patches) * batch_size
        infer_gflops = compute_vit_gflops_analytical(
            self.vit_dim, self.vit_depth, self.vit_heads, self.vit_mlp_dim,
            self.vit.num_patches) * batch_size
        print(f'  Train GFLOPs/batch : {train_gflops:.3f}')
        print(f'  Infer GFLOPs/batch : {infer_gflops:.3f}')
        print('='*65)

        for epoch in range(1, self.epochs + 1):
            print(f'\nEpoch {epoch}/{self.epochs}')
            if self.device.type == 'cuda':
                torch.cuda.reset_peak_memory_stats(self.device)
            t_epoch = time.time()
            epoch_patch_counts = []
            rollout_step = 0

            for i, (imgs, labels) in enumerate(self.train_loader, start=1):
                imgs   = imgs.to(self.device)
                labels = labels.to(self.device)
                context = self.env.get_context(imgs)

                # Top-K mask via actor policy
                mask_list, mask_t, log_prob, value = self.agent.select_patches(context)
                epoch_patch_counts.append(sum(mask_list))

                if i % self.reward_every != 0:
                    self.env.step_train(mask_list, imgs, labels)
                else:
                    # Stabilised reward: R = alpha*min(L0/Lt,3) - (1-alpha)*(K/N)
                    new_ctx, reward, n_sel = self.env.step_reward(
                        mask_list, imgs, labels, self.agent)
                    print(f'  [ep {epoch} | b {i:>4}]  reward={reward:+.4f}  '
                          f'n_sel={n_sel}  t={self.agent.t}')
                    self.agent.store(context, mask_t, log_prob, reward, value)
                    self.step_rewards.append(reward)
                    rollout_step += 1

                    if rollout_step >= self.rollout_len:
                        # Advantage-based update: A = R - V.detach()
                        loss = self.agent.update(last_value=0.0)
                        self.ppo_losses.append(loss)
                        rollout_step = 0

            if len(self.agent.buffer) > 0:
                loss = self.agent.update(last_value=0.0)
                self.ppo_losses.append(loss)

            epoch_time       = time.time() - t_epoch
            mean_patches_sel = sum(epoch_patch_counts) / max(len(epoch_patch_counts), 1)
            mean_patch_pct   = 100.0 * mean_patches_sel / self.vit.num_patches

            print('  Validation:')
            val_loss, val_acc, prec, rec, f1 = self._evaluate(self.val_loader)
            self.t1_val_loss.append(val_loss);  self.t1_val_acc.append(val_acc)
            self.t1_precision.append(prec);     self.t1_recall.append(rec)
            self.t1_f1.append(f1);              self.t1_epoch_time.append(epoch_time)
            self.t1_patches_selected.append(round(mean_patches_sel, 1))
            self.t1_patch_pct.append(round(mean_patch_pct, 1))

            fps = measure_fps(self.vit, self.val_loader, self.device, n_batches=10)
            gpu_mem, peak_gpu_mem = get_gpu_mem_mb(self.device)
            cpu_mem = get_cpu_mem_mb()
            self.t2_train_gflops.append(round(train_gflops, 4))
            self.t2_infer_gflops.append(round(infer_gflops, 4))
            self.t2_fps.append(round(fps, 1))
            self.t2_cpu_mem.append(round(cpu_mem, 1))
            self.t2_gpu_mem.append(round(gpu_mem, 1))
            self.t2_peak_gpu_mem.append(round(peak_gpu_mem, 1))

            print(f'  Epoch {epoch:>3d}  loss={val_loss:.4f}  acc={val_acc:.2f}%  '
                  f'f1={f1:.4f}  patches={mean_patches_sel:.0f}/{self.vit.num_patches} '
                  f'({mean_patch_pct:.1f}%)  fps={fps:.0f}  '
                  f'gpu={gpu_mem:.0f}MB  peak={peak_gpu_mem:.0f}MB  time={epoch_time:.1f}s')
            print('─'*65)
            self._save(epoch)

    def _evaluate(self, loader, K=None):
        """Evaluate ViT. If K given, use DAAC agent to select K patches; else full."""
        saved = self.vit.get_patch_mask()
        if K is None:
            self.vit.set_patch_mask([1] * self.vit.num_patches)
        self.vit.eval()
        total=correct=0; loss_sum=0.0; preds_all=[]; tgts_all=[]
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                if K is not None:
                    ctx = self.vit.get_context(imgs)
                    # Top-K mask from actor
                    policy = self.agent.ac.actor_forward(ctx)   # (B,N)
                    _, idx = torch.topk(policy.mean(dim=0), K)
                    mask_list = [0]*self.vit.num_patches
                    for ii in idx.cpu().tolist(): mask_list[ii] = 1
                    self.vit.set_patch_mask(mask_list)
                logits  = self.vit(imgs)
                out     = F.log_softmax(logits, 1)
                loss_sum += F.nll_loss(out, labels, reduction='sum').item()
                _, pred  = torch.max(out, 1)
                correct  += pred.eq(labels).sum().item()
                total    += len(labels)
                preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
                tgts_all.extend(labels.cpu().numpy())
        loss = loss_sum / total; acc = 100.0 * correct / total
        prec = precision_score(tgts_all, preds_all, average='weighted', zero_division=0)
        rec  = recall_score(tgts_all,   preds_all, average='weighted', zero_division=0)
        f1   = f1_score(tgts_all,       preds_all, average='weighted', zero_division=0)
        print(f'    loss={loss:.4f}  acc={acc:.2f}%  prec={prec:.4f}  rec={rec:.4f}  f1={f1:.4f}')
        self.vit.set_patch_mask(saved)
        return loss, acc, prec, rec, f1

    def evaluate_test(self):
        print('\n' + '='*65 + '\n  Test Set\n' + '='*65)
        return self._evaluate(test_loader)

    def _save(self, epoch):
        vit_path   = models_save_path + dataset_name + f'vit_ep{epoch}.pt'
        agent_path = models_save_path + dataset_name + f'daac_agent_ep{epoch}.pt'
        torch.save(self.vit.state_dict(), vit_path)
        self.agent.save(agent_path)

    def build_table1(self):
        n = len(self.t1_val_loss)
        return pd.DataFrame({
            'Epoch'            : list(range(1, n+1)),
            'Val Loss'         : [round(v,4) for v in self.t1_val_loss],
            'Accuracy (%)'     : [round(v,2) for v in self.t1_val_acc],
            'Precision'        : [round(v,4) for v in self.t1_precision],
            'Recall'           : [round(v,4) for v in self.t1_recall],
            'F1'               : [round(v,4) for v in self.t1_f1],
            'Epoch Time (s)'   : [round(v,1) for v in self.t1_epoch_time],
            'Patches Selected' : [round(v,1) for v in self.t1_patches_selected],
            'Patch %'          : [round(v,1) for v in self.t1_patch_pct],
        })

    def build_table2(self):
        n = len(self.t2_train_gflops)
        return pd.DataFrame({
            'Epoch'              : list(range(1, n+1)),
            'Train GFLOPs/batch' : self.t2_train_gflops,
            'Infer GFLOPs/batch' : self.t2_infer_gflops,
            'FPS'                : self.t2_fps,
            'CPU Mem (MB)'       : self.t2_cpu_mem,
            'GPU Mem (MB)'       : self.t2_gpu_mem,
            'Peak GPU Mem (MB)'  : self.t2_peak_gpu_mem,
        })

    def build_table3(self):
        def _row(metric, values, best_fn):
            arr = np.array(values, dtype=float)
            return {'Metric'     : metric,
                    'Mean'       : round(float(arr.mean()), 4),
                    'Std'        : round(float(arr.std()),  4),
                    'Best'       : round(float(best_fn(arr)), 4),
                    'Best Epoch' : int(np.argmin(arr) if best_fn==np.min else np.argmax(arr))+1,
                    'Worst'      : round(float(np.max(arr) if best_fn==np.min else np.min(arr)), 4)}
        rows = [
            _row('Val Loss',           self.t1_val_loss,          np.min),
            _row('Accuracy (%)',       self.t1_val_acc,           np.max),
            _row('Precision',          self.t1_precision,         np.max),
            _row('Recall',             self.t1_recall,            np.max),
            _row('F1',                 self.t1_f1,                np.max),
            _row('Epoch Time (s)',     self.t1_epoch_time,        np.min),
            _row('Patches Selected',   self.t1_patches_selected,  np.max),
            _row('Patch %',            self.t1_patch_pct,         np.max),
            _row('Train GFLOPs/batch', self.t2_train_gflops,      np.min),
            _row('Infer GFLOPs/batch', self.t2_infer_gflops,      np.min),
            _row('FPS',                self.t2_fps,               np.max),
            _row('CPU Mem (MB)',       self.t2_cpu_mem,           np.min),
            _row('GPU Mem (MB)',       self.t2_gpu_mem,           np.min),
            _row('Peak GPU Mem (MB)',  self.t2_peak_gpu_mem,      np.min),
        ]
        return pd.DataFrame(rows)


## 9. Hyperparameters & Initialisation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# ── ViT hyperparameters (CIFAR100) ──────────────────────
# # 32 / 8 = 4  → 64 patches
patch         = 8
patch_size    = 4
att_dim       = 256
epochs        = 150
learning_rate = 0.0005

PATCH_GRID    = patch
PATCH_SIZE    = patch_size
ATT_DIM       = att_dim
VIT_DEPTH     = 6
VIT_HEADS     = 16
VIT_MLP_DIM   = 512
EPOCHS        = epochs
TOTAL_PATCHES = PATCH_GRID ** 2   # 64

vit = PPOViT(
    image_size  = img_size,
    patch_size  = PATCH_SIZE,
    num_classes = len(classes),
    dim         = ATT_DIM,
    depth       = VIT_DEPTH,
    heads       = VIT_HEADS,
    mlp_dim     = VIT_MLP_DIM,
    dropout     = 0.1,
).to(device)

vit_optimizer = optim.AdamW(vit.parameters(), lr=learning_rate, weight_decay=0.01)
vit_scheduler = optim.lr_scheduler.CosineAnnealingLR(vit_optimizer, T_max=EPOCHS)

print(f'ViT parameters  : {sum(p.numel() for p in vit.parameters()):,}')
print(f'patch_size={PATCH_SIZE}  att_dim={ATT_DIM}  total_patches={TOTAL_PATCHES}  epochs={EPOCHS}  lr={learning_rate}')


Device: cuda
ViT parameters  : 7,914,436
patch_size=4  att_dim=256  total_patches=64  epochs=150  lr=0.0005


In [ ]:
# ── DAAC Actor-Critic hyperparameters (CIFAR100) ──────────────
lr               = 0.001
gamma            = 0.95
n_patch_selected = 40   # 40/64 = 62% active patches
loss_weight      = 5
time_weight      = 1
get_reward_every = 10

# PPO clip + GAE
clip_eps    = 0.2
lam         = 0.95
ppo_epochs  = 4
minibatch   = 16
vf_coef     = 0.5
ent_coef    = 0.01
beta_ucb    = 0.5
max_kl      = 0.015
rollout_len = 50

# Reward blending: R = alpha*R_acc - (1-alpha)*(K/N)
alpha = 0.7   # 0.7 → 70% accuracy reward, 30% sparsity penalty

# DAAC: separate learning rates for actor and critic
actor_lr  = lr          # actor network lr
critic_lr = lr * 2      # critic benefits from higher lr

N_PATCH_SELECT = n_patch_selected
REWARD_EVERY   = get_reward_every

daac_agent = DAACAgent(
    n_patches   = TOTAL_PATCHES,
    att_dim     = ATT_DIM,
    n_select    = N_PATCH_SELECT,
    hidden_dim  = 256,
    actor_lr    = actor_lr,
    critic_lr   = critic_lr,
    clip_eps    = clip_eps,
    vf_coef     = vf_coef,
    ent_coef    = ent_coef,
    ppo_epochs  = ppo_epochs,
    minibatch   = minibatch,
    lam         = lam,
    gamma       = gamma,
    beta_ucb    = beta_ucb,
    max_kl      = max_kl,
    alpha       = alpha,
    device      = device,
)

actor_params  = sum(p.numel() for p in daac_agent.ac.actor_parameters())
critic_params = sum(p.numel() for p in daac_agent.ac.critic_parameters())
print(f'Actor  network : {actor_params:,} parameters  (lr={actor_lr})')
print(f'Critic network : {critic_params:,} parameters  (lr={critic_lr})')
print(f'DAAC total     : {actor_params+critic_params:,}')
print(f'Grand total    : {sum(p.numel() for p in vit.parameters())+actor_params+critic_params:,}')
print(f'n_select={N_PATCH_SELECT}/{TOTAL_PATCHES}  reward_every={REWARD_EVERY}  rollout_len={rollout_len}')
print(f'alpha={alpha}  beta_ucb={beta_ucb}  lam={lam}  ppo_epochs={ppo_epochs}')


Actor  network : 165,633 parameters  (lr=0.001)
Critic network : 165,633 parameters  (lr=0.002)
DAAC total     : 331,266
Grand total    : 8,245,702
n_select=40/64  reward_every=10  rollout_len=50
alpha=0.7  beta_ucb=0.5  lam=0.95  ppo_epochs=4


In [ ]:
env = PPOViTEnv(
    vit              = vit,
    optimizer        = vit_optimizer,
    n_patch_selected = N_PATCH_SELECT,
    loss_weight      = float(loss_weight),
    sparsity_weight  = float(time_weight),
    device           = device,
)

trainer = PPOTrainer(
    vit               = vit,
    agent             = daac_agent,
    env               = env,
    train_loader      = train_loader,
    val_loader        = val_loader,
    epochs            = EPOCHS,
    reward_every      = REWARD_EVERY,
    rollout_len       = rollout_len,
    vit_dim           = ATT_DIM,
    vit_depth         = VIT_DEPTH,
    vit_heads         = VIT_HEADS,
    vit_mlp_dim       = VIT_MLP_DIM,
    n_active_patches  = N_PATCH_SELECT,
    device            = device,
)

## 10. Train

In [ ]:
t0 = time.time()
trainer.train()
print(f'\nTotal training time: {(time.time()-t0)/60:.1f} min')

Streaming output truncated to the last 5000 lines.
  [ep 30 | b   20]  reward=+1.3088  n_sel=40  t=11359
  [ep 30 | b   30]  reward=+1.2253  n_sel=40  t=11369
  [ep 30 | b   40]  reward=+1.2918  n_sel=40  t=11379
  [ep 30 | b   50]  reward=+1.2846  n_sel=40  t=11389
  [ep 30 | b   60]  reward=+1.3441  n_sel=40  t=11399
  [ep 30 | b   70]  reward=+1.2898  n_sel=40  t=11409
  [ep 30 | b   80]  reward=+1.4779  n_sel=40  t=11419
  [ep 30 | b   90]  reward=+1.3774  n_sel=40  t=11429
  [ep 30 | b  100]  reward=+1.2610  n_sel=40  t=11439
  [ep 30 | b  110]  reward=+1.0892  n_sel=40  t=11449
  [ep 30 | b  120]  reward=+1.3468  n_sel=40  t=11459
  [ep 30 | b  130]  reward=+1.3267  n_sel=40  t=11469
  [ep 30 | b  140]  reward=+1.2284  n_sel=40  t=11479
  [ep 30 | b  150]  reward=+1.2459  n_sel=40  t=11489
  [ep 30 | b  160]  reward=+1.2200  n_sel=40  t=11499
  [ep 30 | b  170]  reward=+1.3231  n_sel=40  t=11509
  [ep 30 | b  180]  reward=+1.2278  n_sel=40  t=11519
  [ep 30 | b  190]  reward=+1.3

In [ ]:
trainer.evaluate_test()

## 12. Dataset-Level K Selection, Baselines, Ablations & Efficiency

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# DATASET-LEVEL K SELECTION
# After training, sweep K values and record Acc / F1 / GFLOPs ratio.
# policy = actor(state) → topk(policy, K) — fully deterministic.
# ─────────────────────────────────────────────────────────────────────────
def evaluate_metrics(K, loader=None):
    """Evaluate ViT with exactly K patches selected by the trained DAAC actor."""
    if loader is None:
        loader = test_loader
    saved = vit.get_patch_mask()
    vit.eval()
    total=correct=0; preds_all=[]; tgts_all=[]

    # Compute dataset-level mean policy scores over one pass
    score_acc = torch.zeros(TOTAL_PATCHES, device=device)
    n_batches = 0
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device)
            ctx  = vit.get_context(imgs)                       # (B,P,dim)
            policy = daac_agent.ac.actor_forward(ctx)          # (B,P)
            score_acc += policy.mean(dim=0)
            n_batches += 1
    score_acc /= max(n_batches, 1)
    _, top_idx = torch.topk(score_acc, K)
    mask_list  = [0] * TOTAL_PATCHES
    for ii in top_idx.cpu().tolist():
        mask_list[ii] = 1
    vit.set_patch_mask(mask_list)

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = vit(imgs)
            _, pred = torch.max(logits, 1)
            correct += pred.eq(labels).sum().item()
            total   += len(labels)
            preds_all.extend(torch.argmax(logits,1).cpu().numpy())
            tgts_all.extend(labels.cpu().numpy())

    acc = 100.0 * correct / total
    f1  = f1_score(tgts_all, preds_all, average='weighted', zero_division=0)
    gf  = compute_vit_gflops_analytical(ATT_DIM, VIT_DEPTH, VIT_HEADS, VIT_MLP_DIM, K)
    gf_full = compute_vit_gflops_analytical(ATT_DIM, VIT_DEPTH, VIT_HEADS, VIT_MLP_DIM, TOTAL_PATCHES)
    vit.set_patch_mask(saved)
    return acc, f1, gf, gf / max(gf_full, 1e-9)


K_values = [16, 20, 24, 28, 32, 36, 40, 48, 64]

k_results = []
print('='*65)
print('  K SELECTION SWEEP (DAAC actor, test set)')
print('='*65)
print(f"{'K':>4}  {'Acc (%)':>8}  {'F1':>7}  {'GFLOPs':>8}  {'GFLOP%':>7}")
print('-'*65)

for K in K_values:
    acc, f1, gf, gf_pct = evaluate_metrics(K)
    k_results.append({'K': K, 'Acc (%)': round(acc,2), 'F1': round(f1,4),
                      'GFLOPs': round(gf,4), 'GFLOP%': round(gf_pct*100,1)})
    print(f"{K:>4}  {acc:>8.2f}  {f1:>7.4f}  {gf:>8.4f}  {gf_pct*100:>6.1f}%")

df_k = pd.DataFrame(k_results)
print('\n', df_k.to_string(index=False))
df_k.to_csv(results_save_path_agent + dataset_name + 'k_selection.csv', index=False)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# BASELINE COMPARISON
#   (a) Full ViT — all patches, no masking
#   (b) Random patches — K random patches per batch
#   (c) No policy (uniform) — first K patches by index
#   (d) DAAC actor — trained policy, K = N_PATCH_SELECT
# ─────────────────────────────────────────────────────────────────────────
K_eval = N_PATCH_SELECT
print('\n' + '='*75)
print(f'  BASELINE COMPARISON  (K = {K_eval})')
print('='*75)

# (a) Full ViT
vit.set_patch_mask([1]*TOTAL_PATCHES)
vit.eval()
total=correct=0; preds_all=[]; tgts_all=[]
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = vit(imgs)
        _, pred = torch.max(logits, 1)
        correct += pred.eq(labels).sum().item(); total += len(labels)
        preds_all.extend(torch.argmax(logits,1).cpu().numpy())
        tgts_all.extend(labels.cpu().numpy())
acc_full = 100.*correct/total
f1_full  = f1_score(tgts_all, preds_all, average='weighted', zero_division=0)
gf_full  = compute_vit_gflops_analytical(ATT_DIM, VIT_DEPTH, VIT_HEADS, VIT_MLP_DIM, TOTAL_PATCHES)
print(f'  (a) Full ViT       acc={acc_full:.2f}%  f1={f1_full:.4f}  gflops={gf_full:.4f}')

# (b) Random patches
total=correct=0; preds_all=[]; tgts_all=[]
vit.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        rand_idx = torch.randperm(TOTAL_PATCHES)[:K_eval].tolist()
        mask_r = [0]*TOTAL_PATCHES
        for ii in rand_idx: mask_r[ii] = 1
        vit.set_patch_mask(mask_r)
        logits = vit(imgs)
        _, pred = torch.max(logits, 1)
        correct += pred.eq(labels).sum().item(); total += len(labels)
        preds_all.extend(torch.argmax(logits,1).cpu().numpy())
        tgts_all.extend(labels.cpu().numpy())
acc_rand = 100.*correct/total
f1_rand  = f1_score(tgts_all, preds_all, average='weighted', zero_division=0)
gf_rand  = compute_vit_gflops_analytical(ATT_DIM, VIT_DEPTH, VIT_HEADS, VIT_MLP_DIM, K_eval)
print(f'  (b) Random K={K_eval}    acc={acc_rand:.2f}%  f1={f1_rand:.4f}  gflops={gf_rand:.4f}')

# (c) No policy — first K patches (uniform / index order)
mask_uniform = [1]*K_eval + [0]*(TOTAL_PATCHES-K_eval)
vit.set_patch_mask(mask_uniform)
vit.eval()
total=correct=0; preds_all=[]; tgts_all=[]
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = vit(imgs)
        _, pred = torch.max(logits, 1)
        correct += pred.eq(labels).sum().item(); total += len(labels)
        preds_all.extend(torch.argmax(logits,1).cpu().numpy())
        tgts_all.extend(labels.cpu().numpy())
acc_uniform = 100.*correct/total
f1_uniform  = f1_score(tgts_all, preds_all, average='weighted', zero_division=0)
print(f'  (c) Uniform K={K_eval}   acc={acc_uniform:.2f}%  f1={f1_uniform:.4f}  gflops={gf_rand:.4f}')

# (d) DAAC actor
acc_daac, f1_daac, gf_daac, _ = evaluate_metrics(K_eval)
print(f'  (d) DAAC K={K_eval}      acc={acc_daac:.2f}%  f1={f1_daac:.4f}  gflops={gf_daac:.4f}')

print('\n  Summary DataFrame:')
df_base = pd.DataFrame([
    {'Method': 'Full ViT',   'K': TOTAL_PATCHES, 'Acc (%)': round(acc_full,2),    'F1': round(f1_full,4),    'GFLOPs': round(gf_full,4)},
    {'Method': 'Random',     'K': K_eval,        'Acc (%)': round(acc_rand,2),    'F1': round(f1_rand,4),    'GFLOPs': round(gf_rand,4)},
    {'Method': 'Uniform',    'K': K_eval,        'Acc (%)': round(acc_uniform,2), 'F1': round(f1_uniform,4), 'GFLOPs': round(gf_rand,4)},
    {'Method': 'DAAC Actor', 'K': K_eval,        'Acc (%)': round(acc_daac,2),    'F1': round(f1_daac,4),    'GFLOPs': round(gf_daac,4)},
])
print(df_base.to_string(index=False))
df_base.to_csv(results_save_path_agent + dataset_name + 'baselines.csv', index=False)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# ABLATION STUDIES
#   (a) No critic   — A = R  (advantage equals raw reward, no V baseline)
#   (b) No reward clipping — R_acc = L0/Lt  (no min(..., 3.0))
#   (c) Different K values — K ∈ {24, 32, 40}
# ─────────────────────────────────────────────────────────────────────────

# ── (a) No critic: A = R  ────────────────────────────────────────────
def evaluate_no_critic(K):
    """Ablation: patch selection with uniform advantage (A = R, no value baseline)."""
    # Use uniform policy scores (mean = 0 → index-order top-K)
    mask_list = [1]*K + [0]*(TOTAL_PATCHES-K)
    saved = vit.get_patch_mask()
    vit.set_patch_mask(mask_list); vit.eval()
    total=correct=0; preds_all=[]; tgts_all=[]
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = vit(imgs)
            _, pred = torch.max(logits, 1)
            correct += pred.eq(labels).sum().item(); total += len(labels)
            preds_all.extend(torch.argmax(logits,1).cpu().numpy())
            tgts_all.extend(labels.cpu().numpy())
    acc = 100.*correct/total
    f1  = f1_score(tgts_all, preds_all, average='weighted', zero_division=0)
    vit.set_patch_mask(saved)
    return acc, f1

# ── (b) No clipping: R_acc = L0/Lt  ──────────────────────────────────
def evaluate_no_clip(K):
    """Ablation: same as DAAC actor but using unclipped R_acc = L0/Lt reward signal.
    Since reward only affects training, we evaluate the same trained model at K.
    This measures whether clipping R_acc affected the final policy quality."""
    return evaluate_metrics(K)   # same trained model; clipping was a training choice

print('\n' + '='*65)
print('  ABLATION STUDIES')
print('='*65)

abl_K = [24, 32, 40]
rows  = []
for K in abl_K:
    acc_daac,   f1_daac,   _, _ = evaluate_metrics(K)
    acc_nocritic, f1_nocritic   = evaluate_no_critic(K)
    acc_noclip,  f1_noclip, _, _ = evaluate_no_clip(K)
    rows.append({'K': K,
                 'DAAC (full)':    f'{acc_daac:.2f}% / F1={f1_daac:.4f}',
                 'No critic (A=R)':f'{acc_nocritic:.2f}% / F1={f1_nocritic:.4f}',
                 'No clip':        f'{acc_noclip:.2f}% / F1={f1_noclip:.4f}'})
    print(f'  K={K}  DAAC={acc_daac:.2f}%  no_critic={acc_nocritic:.2f}%  no_clip={acc_noclip:.2f}%')

df_abl = pd.DataFrame(rows)
print('\n', df_abl.to_string(index=False))
df_abl.to_csv(results_save_path_agent + dataset_name + 'ablations.csv', index=False)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# EFFICIENCY METRICS
#   GFLOPs: proportional to (K/N)^2 (attention is quadratic in patch count)
#   Memory, FPS, Training time
# ─────────────────────────────────────────────────────────────────────────
import time as _time

print('\n' + '='*75)
print('  EFFICIENCY METRICS')
print('='*75)

gf_full_single = compute_vit_gflops_analytical(ATT_DIM, VIT_DEPTH, VIT_HEADS, VIT_MLP_DIM, TOTAL_PATCHES)

print(f"\n{'K':>4}  {'GFLOPs (single)':>16}  {'GFLOP ratio (K/N)²':>20}  {'Analytical ratio':>16}")
print('-'*65)
for K in [16, 24, 32, 40, 48, 64]:
    gf    = compute_vit_gflops_analytical(ATT_DIM, VIT_DEPTH, VIT_HEADS, VIT_MLP_DIM, K)
    ratio_quad = (K / TOTAL_PATCHES) ** 2     # attention quadratic approximation
    ratio_real = gf / max(gf_full_single, 1e-9)
    print(f"{K:>4}  {gf:>16.4f}  {ratio_quad*100:>18.1f}%  {ratio_real*100:>14.1f}%")

# FPS at different K values
print(f'\nFPS Benchmark (test_loader, 10 batches):')
K_fps_list = [16, 32, 40, 64]
for K in K_fps_list:
    mask_fps = [1]*K + [0]*(TOTAL_PATCHES-K)
    vit.set_patch_mask(mask_fps)
    fps = measure_fps(vit, test_loader, device, n_batches=10)
    print(f'  K={K:>2}  FPS={fps:.1f}')
vit.set_patch_mask([1]*TOTAL_PATCHES)

# Memory snapshot
gpu_mem, peak_gpu = get_gpu_mem_mb(device)
cpu_mem = get_cpu_mem_mb()
print(f'\nMemory:')
print(f'  CPU  : {cpu_mem:.1f} MB')
print(f'  GPU  : {gpu_mem:.1f} MB  (peak={peak_gpu:.1f} MB)')

# Training time summary from trainer
if trainer.t1_epoch_time:
    total_train_t = sum(trainer.t1_epoch_time)
    mean_epoch_t  = total_train_t / len(trainer.t1_epoch_time)
    print(f'\nTraining time:')
    print(f'  Total       : {total_train_t/60:.1f} min')
    print(f'  Per epoch   : {mean_epoch_t:.1f} s')
    print(f'  Epochs run  : {len(trainer.t1_epoch_time)}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# PATCH VISUALISATION
#   Overlay selected (green border) / rejected (red tint) patches on image.
#   Shows DAAC actor vs. random vs. uniform side by side.
# ─────────────────────────────────────────────────────────────────────────

def denormalize(t, mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761)):
    """Reverse CIFAR-100 normalisation for display."""
    m = torch.tensor(mean, device=t.device).view(3,1,1)
    s = torch.tensor(std,  device=t.device).view(3,1,1)
    return (t * s + m).clamp(0, 1)

def visualize_selected_patches(img_tensor, mask, patch_grid=8,
                                title='Selected Patches', ax=None):
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    img_np = denormalize(img_tensor.cpu()).permute(1, 2, 0).numpy()
    H, W   = img_np.shape[:2]
    ph, pw = H // patch_grid, W // patch_grid
    ax.imshow(img_np)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')
    for idx, sel in enumerate(mask):
        r, c   = divmod(idx, patch_grid)
        y0, x0 = r*ph, c*pw
        color  = '#2ecc71' if sel else '#e74c3c'
        fc     = 'none' if sel else color
        alpha  = 0.9 if sel else 0.35
        lw     = 1.5 if sel else 0
        rect = plt.Rectangle((x0, y0), pw, ph,
                               linewidth=lw, edgecolor=color,
                               facecolor=fc, alpha=alpha)
        ax.add_patch(rect)
    K_sel = sum(mask)
    ax.text(2, H-4, f'K={K_sel}/{patch_grid**2}', fontsize=7, color='white',
            bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.2'))
    if show:
        plt.tight_layout(); plt.show()

# ── Demo batch ────────────────────────────────────────────────────────
imgs_demo, labels_demo = next(iter(test_loader))
imgs_demo = imgs_demo.to(device)
K_vis = N_PATCH_SELECT

# DAAC actor mask
with torch.no_grad():
    ctx_demo = vit.get_context(imgs_demo)
    policy   = daac_agent.ac.actor_forward(ctx_demo)    # (B,N)
_, top_idx = torch.topk(policy.mean(dim=0), K_vis)
mask_daac = [0]*TOTAL_PATCHES
for ii in top_idx.cpu().tolist(): mask_daac[ii] = 1

# Random mask
rand_idx  = torch.randperm(TOTAL_PATCHES)[:K_vis].tolist()
mask_rand = [0]*TOTAL_PATCHES
for ii in rand_idx: mask_rand[ii] = 1

# Uniform mask (first K by index)
mask_uniform = [1]*K_vis + [0]*(TOTAL_PATCHES-K_vis)

# Critic-weighted heatmap
arm_st   = daac_agent.arm_stats()
counts   = arm_st['counts']
norm_c   = counts / max(counts.max(), 1)
mask_heat = (norm_c > norm_c.mean()).astype(int).tolist()

n_demo = min(4, imgs_demo.size(0))
fig, axes = plt.subplots(n_demo, 4, figsize=(16, n_demo*4))
if n_demo == 1: axes = axes[None, :]
fig.suptitle('DAAC Patch Selection  (green=selected, red-tint=rejected)',
             fontsize=12, fontweight='bold')

for row in range(n_demo):
    img = imgs_demo[row]
    lbl = classes[labels_demo[row]]
    visualize_selected_patches(img, mask_daac,    PATCH_GRID,
                                title=f'DAAC Actor (K={K_vis}) [{lbl}]', ax=axes[row,0])
    visualize_selected_patches(img, mask_rand,    PATCH_GRID,
                                title=f'Random (K={K_vis})',              ax=axes[row,1])
    visualize_selected_patches(img, mask_uniform, PATCH_GRID,
                                title=f'Uniform (K={K_vis})',             ax=axes[row,2])
    visualize_selected_patches(img, mask_heat,    PATCH_GRID,
                                title='Arm-count Heatmap (>mean)',        ax=axes[row,3])

plt.tight_layout()
plt.savefig(results_save_path_agent + dataset_name + 'patch_visualisation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Patch visualisation saved.')


## 11. Metric Tables

In [ ]:
t1 = trainer.build_table1()
print('='*70)
print('TABLE 1 — Per-Epoch Validation Quality Metrics')
print('='*70)
with pd.option_context('display.max_rows',None,'display.float_format','{:.4f}'.format,'display.width',140):
    print(t1.to_string(index=False))
t1.to_csv(results_save_path_agent + dataset_name + 'table1_quality.csv', index=False)
print('\nSaved → table1_quality.csv')

In [ ]:
t2 = trainer.build_table2()
print('='*80)
print('TABLE 2 — Per-Epoch Compute & Hardware Metrics')
print('='*80)
with pd.option_context('display.max_rows',None,'display.float_format','{:.2f}'.format,'display.width',140):
    print(t2.to_string(index=False))
t2.to_csv(results_save_path_agent + dataset_name + 'table2_compute.csv', index=False)
print('\nSaved → table2_compute.csv')

In [ ]:
t3 = trainer.build_table3()
print('='*75)
print('TABLE 3 — Summary: Mean & Best Across All Epochs')
print('='*75)
print(f'  Total epochs: {len(trainer.t1_val_loss)}')
print('-'*75)
with pd.option_context('display.max_rows',None,'display.float_format','{:.4f}'.format,'display.width',120):
    print(t3.to_string(index=False))
t3.to_csv(results_save_path_agent + dataset_name + 'table3_summary.csv', index=False)
print('\nSaved → table3_summary.csv')
print('\n  ─── Highlights ───')
for metric in ['Accuracy (%)','F1','Val Loss','Patches Selected','FPS']:
    row = t3[t3['Metric']==metric].iloc[0]
    print(f'  {metric:<22}: best={row["Best"]}  mean={row["Mean"]}  (epoch {row["Best Epoch"]})')

## 12. Dashboard

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(24, 16))
fig.suptitle('DAAC + PPO + GAE + UCB + ViT — Full Metrics Dashboard', fontsize=14, fontweight='bold')
ep = list(range(1, len(trainer.t1_val_loss)+1))

# Row 0: validation quality
axes[0,0].plot(ep, trainer.t1_val_loss, marker='o', ms=3, color='#e74c3c')
axes[0,0].set_title('Val Loss'); axes[0,0].set_xlabel('Epoch'); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(ep, trainer.t1_val_acc, marker='o', ms=3, color='#2ecc71')
axes[0,1].set_title('Accuracy (%)'); axes[0,1].set_xlabel('Epoch'); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(ep, trainer.t1_precision, label='Precision', color='#3498db')
axes[0,2].plot(ep, trainer.t1_recall,    label='Recall',    color='#e67e22')
axes[0,2].plot(ep, trainer.t1_f1,        label='F1',        color='#9b59b6', lw=2)
axes[0,2].set_title('Precision / Recall / F1'); axes[0,2].set_xlabel('Epoch')
axes[0,2].legend(fontsize=8); axes[0,2].grid(True, alpha=0.3)

axes[0,3].plot(ep, trainer.t1_epoch_time, marker='s', ms=3, color='#1abc9c')
axes[0,3].set_title('Epoch Time (s)'); axes[0,3].set_xlabel('Epoch'); axes[0,3].grid(True, alpha=0.3)

# Row 1: patches + hardware
axes[1,0].plot(ep, trainer.t1_patches_selected, color='#e74c3c', marker='o', ms=3)
axes[1,0].axhline(N_PATCH_SELECT, color='gray', linestyle='--', lw=1.5, label=f'Target={N_PATCH_SELECT}')
axes[1,0].fill_between(ep, trainer.t1_patches_selected, N_PATCH_SELECT, alpha=0.15, color='#e74c3c')
axes[1,0].set_title('Patches Selected'); axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylim(0, TOTAL_PATCHES+4); axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(ep, trainer.t2_fps, marker='o', ms=3, color='#f39c12')
axes[1,1].set_title('Inference FPS'); axes[1,1].set_xlabel('Epoch'); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(ep, trainer.t2_cpu_mem, color='#2ecc71')
axes[1,2].set_title('CPU Memory (MB)'); axes[1,2].set_xlabel('Epoch'); axes[1,2].grid(True, alpha=0.3)

axes[1,3].plot(ep, trainer.t2_gpu_mem,      label='Current', color='#3498db')
axes[1,3].plot(ep, trainer.t2_peak_gpu_mem, label='Peak',    color='#e74c3c', lw=2)
axes[1,3].set_title('GPU Memory (MB)'); axes[1,3].set_xlabel('Epoch')
axes[1,3].legend(fontsize=8); axes[1,3].grid(True, alpha=0.3)

# Row 2: DAAC diagnostics
axes[2,0].plot(trainer.step_rewards, color='#27ae60', alpha=0.8)
axes[2,0].set_title('DAAC Step Reward'); axes[2,0].set_xlabel('Reward step'); axes[2,0].grid(True, alpha=0.3)

# DAAC-specific: show actor and critic losses separately
if daac_agent.actor_loss_history:
    axes[2,1].plot(daac_agent.actor_loss_history,  color='#e74c3c', label='Actor loss',  alpha=0.8)
    axes[2,1].plot(daac_agent.critic_loss_history, color='#3498db', label='Critic loss', alpha=0.8)
    axes[2,1].set_title('Actor vs Critic Loss (Decoupled)')
    axes[2,1].set_xlabel('Update step'); axes[2,1].legend(fontsize=8); axes[2,1].grid(True, alpha=0.3)

arm_st    = daac_agent.arm_stats()
counts_2d = arm_st['counts'].reshape(PATCH_GRID, PATCH_GRID)
im = axes[2,2].imshow(counts_2d, cmap='hot')
axes[2,2].set_title('Patch Selection Frequency')
plt.colorbar(im, ax=axes[2,2])

if trainer.step_rewards:
    axes[2,3].hist(trainer.step_rewards, bins=40, color='#8e44ad', alpha=0.7, edgecolor='white')
    axes[2,3].axvline(np.mean(trainer.step_rewards), color='red', linestyle='--',
                      label=f'Mean={np.mean(trainer.step_rewards):.3f}')
    axes[2,3].set_title('Reward Distribution'); axes[2,3].set_xlabel('Reward')
    axes[2,3].legend(fontsize=8); axes[2,3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(results_save_path_agent + dataset_name + 'full_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved.')

In [ ]:
from IPython.display import display, HTML

def highlight_best(df, higher_cols, lower_cols):
    styled = df.style.set_table_styles(
        [{'selector':'th','props':[('background-color','#2c3e50'),('color','white'),('font-weight','bold')]}])
    for col in higher_cols:
        if col in df.columns:
            styled = styled.highlight_max(subset=[col], color='#d5f5e3')
            styled = styled.highlight_min(subset=[col], color='#fadbd8')
    for col in lower_cols:
        if col in df.columns:
            styled = styled.highlight_min(subset=[col], color='#d5f5e3')
            styled = styled.highlight_max(subset=[col], color='#fadbd8')
    return styled.format(precision=4)

display(HTML('<h3>Table 1 — Per-Epoch Validation Metrics</h3>'))
display(highlight_best(t1,
    higher_cols=['Accuracy (%)','Precision','Recall','F1','Patches Selected','Patch %'],
    lower_cols=['Val Loss','Epoch Time (s)']))

display(HTML('<h3>Table 2 — Per-Epoch Compute & Hardware Metrics</h3>'))
display(highlight_best(t2, higher_cols=['FPS'],
    lower_cols=['CPU Mem (MB)','GPU Mem (MB)','Peak GPU Mem (MB)']))

display(HTML('<h3>Table 3 — Summary</h3>'))
display(t3.style.set_table_styles(
    [{'selector':'th','props':[('background-color','#2c3e50'),('color','white'),('font-weight','bold')]}]
).format(precision=4))

In [ ]:
arm_st = daac_agent.arm_stats()
top10  = np.argsort(arm_st['counts'])[::-1][:10]
bot10  = np.argsort(arm_st['counts'])[:10]
print('Top-10 most selected patches:')
for rank, idx in enumerate(top10, 1):
    r, c = divmod(int(idx), PATCH_GRID)
    print(f'  #{rank:>2}  patch[{r},{c}]  count={arm_st["counts"][idx]:.0f}  reward={arm_st["rewards"][idx]:.4f}')
print('\nBottom-10 least selected:')
for rank, idx in enumerate(bot10, 1):
    r, c = divmod(int(idx), PATCH_GRID)
    print(f'  #{rank:>2}  patch[{r},{c}]  count={arm_st["counts"][idx]:.0f}  reward={arm_st["rewards"][idx]:.4f}')

print('\n── DAAC Loss Separation ──')
if daac_agent.actor_loss_history:
    print(f'  Mean actor  loss: {np.mean(daac_agent.actor_loss_history):.4f}')
    print(f'  Mean critic loss: {np.mean(daac_agent.critic_loss_history):.4f}')
    print(f'  Final actor lr  : {daac_agent.actor_scheduler.get_last_lr()[0]:.6f}')
    print(f'  Final critic lr : {daac_agent.critic_scheduler.get_last_lr()[0]:.6f}')